In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [2]:
df = pd.read_csv("C:/Users/Priya/Desktop/Factory-Reallocation-Optimization/data/Nassau Candy Distributor.csv")

print(df.shape)
df.head()

(10194, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [3]:
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
dtype: int64

In [4]:
print("Before :", df.shape)

df.drop_duplicates(inplace=True)
print("After :", df.shape)

Before : (10194, 18)
After : (10194, 18)


In [5]:
df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    format="%d-%m-%Y"
)

df["Ship Date"] = pd.to_datetime(
    df["Ship Date"],
    format="%d-%m-%Y"
)

In [6]:
df["Lead Time"] = (
    df["Ship Date"] -
    df["Order Date"]
).dt.days

In [7]:
df[["Order Date","Ship Date","Lead Time"]].head()

,Order Date,Ship Date,Lead Time
0,2024-01-03,2026-06-30,909
1,2024-01-04,2026-07-01,909
2,2024-01-04,2026-07-01,909
3,2024-01-04,2026-07-01,909
4,2024-01-05,2026-07-05,912


In [8]:
df["Profit Margin"] = (
    df["Gross Profit"] /
    df["Sales"]
)

In [9]:
factory_map = {

"Wonka Bar - Nutty Crunch Surprise":"Lot's O' Nuts",
"Wonka Bar - Fudge Mallows":"Lot's O' Nuts",
"Wonka Bar -Scrumdiddlyumptious":"Lot's O' Nuts",
"Wonka Bar - Milk Chocolate":"Wicked Choccy's",
"Wonka Bar - Triple Dazzle Caramel":"Wicked Choccy's",
"Laffy Taffy":"Sugar Shack",
"SweeTARTS":"Sugar Shack",
"Nerds":"Sugar Shack",
"Fun Dip":"Sugar Shack",
"Fizzy Lifting Drinks":"Sugar Shack",
"Everlasting Gobstopper":"Secret Factory",
"Hair Toffee":"The Other Factory",
"Lickable Wallpaper":"Secret Factory",
"Wonka Gum":"Secret Factory",
"Kazookles":"The Other Factory"
}

df["Factory"] = df["Product Name"].map(factory_map)

In [10]:
df[["Product Name","Factory"]].head(10)

,Product Name,Factory
0,Wonka Bar - Milk Chocolate,Wicked Choccy's
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
5,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
6,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
7,Wonka Bar - Milk Chocolate,Wicked Choccy's
8,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
9,Wonka Bar - Milk Chocolate,Wicked Choccy's


In [11]:
factory_coordinates = {
    "Lot's O' Nuts": (32.881893, -111.768036),
    "Wicked Choccy's": (32.076176, -81.088371),
    "Sugar Shack": (48.119140, -96.181150),
    "Secret Factory": (41.446333, -90.565487),
    "The Other Factory": (35.117500, -89.971107)
}

In [12]:
df["Factory Latitude"] = df["Factory"].map(
    lambda x: factory_coordinates[x][0]
)

df["Factory Longitude"] = df["Factory"].map(
    lambda x: factory_coordinates[x][1]
)

In [13]:
df[[
    "Factory",
    "Factory Latitude",
    "Factory Longitude"
]].head()

,Factory,Factory Latitude,Factory Longitude
0,Wicked Choccy's,32.076176,-81.088371
1,Wicked Choccy's,32.076176,-81.088371
2,Lot's O' Nuts,32.881893,-111.768036
3,Lot's O' Nuts,32.881893,-111.768036
4,Wicked Choccy's,32.076176,-81.088371


In [14]:
city_df = pd.read_csv("C:/Users/Priya/Desktop/Factory-Reallocation-Optimization/data/uscities.csv")

city_df = city_df[
    ['city','state_name','lat','lng']
]

In [15]:
city_df.columns = [
    'City',
    'State/Province',
    'Customer Latitude',
    'Customer Longitude'
]

In [16]:
df = df.merge(
    city_df,
    on=['City','State/Province'],
    how='left'
)

In [17]:
print(df[['Customer Latitude',
          'Customer Longitude']].isnull().sum())

Customer Latitude     1285
Customer Longitude    1285
dtype: int64


In [18]:
missing_cities = df[
    df['Customer Latitude'].isnull()
][['City', 'State/Province']]

print(missing_cities.drop_duplicates().sort_values(['State/Province','City']))

                  City             State/Province
179            Calgary                    Alberta
1294          Edmonton                    Alberta
501          Vancouver           British Columbia
916          Fairfield                Connecticut
2597        Manchester                Connecticut
4410           Milford                Connecticut
2856  Port Saint Lucie                    Florida
85    Saint Petersburg                    Florida
2003     Saint Charles                   Illinois
5179          Winnipeg                   Manitoba
4406           Andover              Massachusetts
477           Franklin              Massachusetts
9600            Canton                   Michigan
8857       Saint Cloud                  Minnesota
5630        Saint Paul                  Minnesota
942      Saint Charles                   Missouri
1695       Saint Louis                   Missouri
4482      Saint Peters                   Missouri
1781           Moncton              New Brunswick


In [19]:
print("Missing unique cities:",
      missing_cities.drop_duplicates().shape[0])

Missing unique cities: 33


In [20]:
df['City'] = (
    df['City']
    .str.strip()
    .str.title()
)

In [21]:
city_df['City'] = (
    city_df['City']
    .str.strip()
    .str.title()
)

In [22]:
df['State/Province'] = (
    df['State/Province']
    .str.strip()
    .str.title()
)

city_df['State/Province'] = (
    city_df['State/Province']
    .str.strip()
    .str.title()
)

In [26]:
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost', 'Lead Time', 'Profit Margin', 'Factory', 'Factory Latitude', 'Factory Longitude', 'Customer Latitude_x', 'Customer Longitude_x', 'Customer Latitude_y', 'Customer Longitude_y']


In [27]:
print(df[['Customer Latitude_x',
          'Customer Latitude_y']].isnull().sum())

Customer Latitude_x    1285
Customer Latitude_y    1266
dtype: int64


In [28]:
print(df[['Customer Longitude_x',
          'Customer Longitude_y']].isnull().sum())

Customer Longitude_x    1285
Customer Longitude_y    1266
dtype: int64


In [29]:
df['Customer Latitude'] = df['Customer Latitude_y']
df['Customer Longitude'] = df['Customer Longitude_y']

In [30]:
df.drop(columns=[
    'Customer Latitude_x',
    'Customer Longitude_x',
    'Customer Latitude_y',
    'Customer Longitude_y'
], inplace=True)

In [31]:
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost', 'Lead Time', 'Profit Margin', 'Factory', 'Factory Latitude', 'Factory Longitude', 'Customer Latitude', 'Customer Longitude']


In [32]:
print(df[['Customer Latitude',
          'Customer Longitude']].isnull().sum())

Customer Latitude     1266
Customer Longitude    1266
dtype: int64


In [33]:
df = df.dropna(subset=[
    "Customer Latitude",
    "Customer Longitude"
]).reset_index(drop=True)

print(df.shape)

(8928, 25)


In [34]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [35]:
df["Shipping Distance (km)"] = df.apply(
    lambda row: haversine(
        row["Factory Latitude"],
        row["Factory Longitude"],
        row["Customer Latitude"],
        row["Customer Longitude"]
    ),
    axis=1
)

In [36]:
df[[
    "Factory",
    "City",
    "Shipping Distance (km)"
]].head()

,Factory,City,Shipping Distance (km)
0,Wicked Choccy's,Houston,1386.426875
1,Wicked Choccy's,Naperville,1244.923655
2,Lot's O' Nuts,Naperville,2298.222856
3,Lot's O' Nuts,Naperville,2298.222856
4,Wicked Choccy's,Philadelphia,1031.103147


In [37]:
df["Shipping Distance (km)"].describe()

count    8928.000000
mean     1943.179438
std      1088.710815
min         1.071762
25%      1032.378759
50%      1649.497940
75%      2897.263115
max      3899.536837
Name: Shipping Distance (km), dtype: float64

In [38]:
df.to_csv(
    "prepared_data.csv",
    index=False
)

In [39]:
print(df.shape)

(8928, 26)


In [40]:
df[[
    "Factory",
    "City",
    "Shipping Distance (km)"
]].head()

,Factory,City,Shipping Distance (km)
0,Wicked Choccy's,Houston,1386.426875
1,Wicked Choccy's,Naperville,1244.923655
2,Lot's O' Nuts,Naperville,2298.222856
3,Lot's O' Nuts,Naperville,2298.222856
4,Wicked Choccy's,Philadelphia,1031.103147


In [41]:
df["Shipping Distance (km)"].describe()

count    8928.000000
mean     1943.179438
std      1088.710815
min         1.071762
25%      1032.378759
50%      1649.497940
75%      2897.263115
max      3899.536837
Name: Shipping Distance (km), dtype: float64

In [42]:
features = [
    'Ship Mode',
    'Division',
    'Region',
    'Product Name',
    'Factory',
    'Shipping Distance (km)',
    'Sales',
    'Units',
    'Cost',
    'Gross Profit',
    'Profit Margin'
]

target = 'Lead Time'

In [43]:
X = df[features].copy()
y = df[target]

In [44]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

categorical_columns = [
    'Ship Mode',
    'Division',
    'Region',
    'Product Name',
    'Factory'
]

for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [46]:
from sklearn.preprocessing import StandardScaler

numeric_columns = [
    'Shipping Distance (km)',
    'Sales',
    'Units',
    'Cost',
    'Gross Profit',
    'Profit Margin'
]

scaler = StandardScaler()

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

In [47]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [48]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, random_state=42)

In [49]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    random_state=42
)

gbr.fit(X_train, y_train)

GradientBoostingRegressor(random_state=42)

In [50]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_model(model, X_test, y_test):

    prediction = model.predict(X_test)

    mse = mean_squared_error(y_test, prediction)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, prediction)
    r2 = r2_score(y_test, prediction)

    return rmse, mae, r2

In [51]:
lr_rmse, lr_mae, lr_r2 = evaluate_model(
    lr,
    X_test,
    y_test
)

In [52]:
rf_rmse, rf_mae, rf_r2 = evaluate_model(
    rf,
    X_test,
    y_test
)

In [53]:
gbr_rmse, gbr_mae, gbr_r2 = evaluate_model(
    gbr,
    X_test,
    y_test
)

In [54]:
results = pd.DataFrame({

    "Model":[
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],

    "RMSE":[
        lr_rmse,
        rf_rmse,
        gbr_rmse
    ],

    "MAE":[
        lr_mae,
        rf_mae,
        gbr_mae
    ],

    "R2 Score":[
        lr_r2,
        rf_r2,
        gbr_r2
    ]

})

results

,Model,RMSE,MAE,R2 Score
0,Linear Regression,266.659353,215.476048,-0.000842
1,Random Forest,283.223586,229.626157,-0.129043
2,Gradient Boosting,265.320981,213.877739,0.009180


In [55]:
best_model_name = results.sort_values(
    by="RMSE"
).iloc[0]["Model"]

print("Best Model :", best_model_name)

Best Model : Gradient Boosting


In [56]:
import joblib

models = {
    "Linear Regression": lr,
    "Random Forest": rf,
    "Gradient Boosting": gbr
}

best_model = models[best_model_name]

joblib.dump(best_model, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [57]:
df.to_csv(
    "prepared_data.csv",
    index=False
)

In [59]:
import joblib

artifacts = {
    "feature_columns": features,
    "categorical_columns": [
        "Ship Mode",
        "Division",
        "Region",
        "Product Name",
        "Factory"
    ],
    "numeric_columns": [
        "Shipping Distance (km)",
        "Sales",
        "Units",
        "Cost",
        "Gross Profit",
        "Profit Margin"
    ],
    "factory_coordinates": factory_coordinates
}

joblib.dump(artifacts, "C:/Users/Priya/Desktop/Factory-Reallocation-Optimization/models/artifacts.pkl")

print("Artifacts saved successfully!")

Artifacts saved successfully!


In [60]:
import numpy as np
import sklearn

print(np.__version__)
print(sklearn.__version__)

2.2.6
1.6.1
